# SmartFlood 2026 LSTM Flood Prediction Model Retraining
## 2-Feature Version: [water_level_m, surge_velocity]

**Purpose:** Retrains the LSTM model after removing rainfall_rate per adviser instruction.
**Architecture:** 2-Layer LSTM with 2 output heads (30-min and 60-min horizon predictions)
**Input features:** water_level_m, surge_velocity
**Sequence length:** 6 timesteps (1 hour of 10-minute readings)

**Output files to download:**
- flood_lstm.onnx → replace server/models/flood_lstm.onnx
- scaler_params.json → replace server/models/scaler_params.json
- test_metrics.json → replace server/models/test_metrics.json
- training_curves.png → replace server/models/training_curves.png
- hydrograph_samples.png → replace server/models/hydrograph_samples.png

In [ ]:
# Cell 1: Install Dependencies
!pip install torch onnx onnxruntime numpy pandas matplotlib scikit-learn -q
print('Done installing.')

In [ ]:
# Cell 2: Imports
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import json
import os
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
import onnx
import onnxruntime as ort
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score

SEED = 42
np.random.seed(SEED)
torch.manual_seed(SEED)
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Using device: {DEVICE}')

In [ ]:
# Cell 3: Hydrological Data Generator
# NOTE: rainfall_rate intentionally EXCLUDED per adviser instruction.
# Features: [water_level_m, surge_velocity]

def generate_hydrograph_scenario(n_points=200, scenario='typhoon'):
    t = np.linspace(0, 1, n_points)
    noise = np.random.normal(0, 0.008, n_points)
    if scenario == 'typhoon':
        base = 0.35
        peak = np.random.uniform(1.55, 1.75)
        rise = np.random.uniform(0.30, 0.50)
        levels = base + (peak - base) * np.where(t < rise, (t / rise) ** 1.8, np.exp(-4.5 * (t - rise)))
    elif scenario == 'moderate':
        base = 0.35
        peak = np.random.uniform(1.05, 1.25)
        rise = np.random.uniform(0.35, 0.55)
        levels = base + (peak - base) * np.where(t < rise, (t / rise) ** 2.0, np.exp(-3.0 * (t - rise)))
    elif scenario == 'flash':
        base = 0.35
        peak = np.random.uniform(1.60, 1.85)
        rise = np.random.uniform(0.15, 0.30)
        levels = base + (peak - base) * np.where(t < rise, (t / rise) ** 1.2, np.exp(-7.0 * (t - rise)))
    elif scenario == 'baseline':
        levels = np.full(n_points, np.random.uniform(0.33, 0.40))
    else:
        base = 0.35
        peak = np.random.uniform(0.55, 0.90)
        rise = np.random.uniform(0.30, 0.50)
        levels = base + (peak - base) * np.where(t < rise, (t / rise) ** 2.2, np.exp(-2.5 * (t - rise)))
    return np.clip(levels + noise, 0.30, 3.50)

def build_dataset(total_scenarios=600):
    all_levels = []
    scenario_mix = [
        ('typhoon',  int(total_scenarios * 0.25)),
        ('flash',    int(total_scenarios * 0.20)),
        ('moderate', int(total_scenarios * 0.25)),
        ('minor',    int(total_scenarios * 0.15)),
        ('baseline', int(total_scenarios * 0.15)),
    ]
    for scen, count in scenario_mix:
        for _ in range(count):
            n_pts = np.random.randint(80, 250)
            all_levels.append(generate_hydrograph_scenario(n_pts, scen))
    return all_levels

print('Generating synthetic hydrograph training data...')
all_scenarios = build_dataset(total_scenarios=600)
all_levels_concat = np.concatenate(all_scenarios)
print(f'Generated {len(all_scenarios)} scenarios, {len(all_levels_concat):,} total data points.')
print(f'Water level range: {all_levels_concat.min():.3f}m to {all_levels_concat.max():.3f}m')

In [ ]:
# Cell 4: Feature Engineering (2 features: water_level_m, surge_velocity)
TIMESTEP_MINUTES  = 10
SEQ_LEN           = 6
HORIZON_30M_STEPS = 3
HORIZON_60M_STEPS = 6
N_FEATURES        = 2

def compute_surge_velocity(levels, clamp_low=-2.0, clamp_high=3.5):
    surge = np.zeros_like(levels)
    surge[1:] = (levels[1:] - levels[:-1]) * 6.0
    surge[0] = 0.0
    return np.clip(surge, clamp_low, clamp_high)

def build_feature_matrix(levels):
    surge = compute_surge_velocity(levels)
    return np.column_stack([levels, surge])

features_raw = build_feature_matrix(all_levels_concat)

feat_min   = features_raw.min(axis=0)
feat_max   = features_raw.max(axis=0)
feat_range = feat_max - feat_min
feat_range = np.where(feat_range == 0, 1.0, feat_range)

features_norm = (features_raw - feat_min) / feat_range
target_norm   = (all_levels_concat - feat_min[0]) / feat_range[0]

print('Feature engineering complete.')
print(f'Feature order : [water_level_m, surge_velocity]')
print(f'feat_min      : {feat_min}')
print(f'feat_max      : {feat_max}')
print(f'feat_range    : {feat_range}')

In [ ]:
# Cell 5: Sliding Window Dataset
def create_sequences(features_n, target_n, seq_len, h30, h60):
    X, y30, y60 = [], [], []
    N = len(target_n)
    for i in range(N - seq_len - h60):
        X.append(features_n[i : i + seq_len])
        y30.append(target_n[i + seq_len + h30 - 1])
        y60.append(target_n[i + seq_len + h60 - 1])
    return np.array(X, dtype=np.float32), np.array(y30, dtype=np.float32), np.array(y60, dtype=np.float32)

X_all, y30_all, y60_all = create_sequences(features_norm, target_norm, SEQ_LEN, HORIZON_30M_STEPS, HORIZON_60M_STEPS)
N = len(X_all)
n_train = int(N * 0.70)
n_val   = int(N * 0.15)

X_train, y30_train, y60_train = X_all[:n_train], y30_all[:n_train], y60_all[:n_train]
X_val,   y30_val,   y60_val   = X_all[n_train:n_train+n_val], y30_all[n_train:n_train+n_val], y60_all[n_train:n_train+n_val]
X_test,  y30_test,  y60_test  = X_all[n_train+n_val:], y30_all[n_train+n_val:], y60_all[n_train+n_val:]

class FloodDataset(Dataset):
    def __init__(self, X, y30, y60):
        self.X = torch.from_numpy(X)
        self.y30 = torch.from_numpy(y30)
        self.y60 = torch.from_numpy(y60)
    def __len__(self): return len(self.X)
    def __getitem__(self, idx): return self.X[idx], self.y30[idx], self.y60[idx]

BATCH_SIZE = 256
train_loader = DataLoader(FloodDataset(X_train, y30_train, y60_train), batch_size=BATCH_SIZE, shuffle=True, drop_last=True)
val_loader   = DataLoader(FloodDataset(X_val,   y30_val,   y60_val),   batch_size=BATCH_SIZE, shuffle=False)
test_loader  = DataLoader(FloodDataset(X_test,  y30_test,  y60_test),  batch_size=BATCH_SIZE, shuffle=False)

print(f'Train: {len(X_train):,} | Val: {len(X_val):,} | Test: {len(X_test):,}')
print(f'X shape: {X_train.shape} = [batch, {SEQ_LEN} timesteps, {N_FEATURES} features]')

In [ ]:
# Cell 6: Model Architecture - 2-Layer LSTM
class FloodLSTM(nn.Module):
    def __init__(self, input_size=2, hidden_size=64, num_layers=2, dropout=0.2):
        super().__init__()
        self.lstm = nn.LSTM(input_size=input_size, hidden_size=hidden_size,
                            num_layers=num_layers, batch_first=True,
                            dropout=dropout if num_layers > 1 else 0.0)
        self.head30 = nn.Sequential(nn.Linear(hidden_size, 32), nn.ReLU(), nn.Linear(32, 1))
        self.head60 = nn.Sequential(nn.Linear(hidden_size, 32), nn.ReLU(), nn.Linear(32, 1))

    def forward(self, x):
        lstm_out, _ = self.lstm(x)
        last = lstm_out[:, -1, :]
        out30 = self.head30(last)
        out60 = self.head60(last)
        return torch.cat([out30, out60], dim=1)

model = FloodLSTM(input_size=N_FEATURES, hidden_size=64, num_layers=2, dropout=0.2).to(DEVICE)
total_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f'FloodLSTM (2-feature) | Params: {total_params:,} | Device: {DEVICE}')
print(model)

In [ ]:
# Cell 7: Training Loop with Early Stopping
EPOCHS   = 80
LR       = 1e-3
PATIENCE = 12

optimizer  = torch.optim.Adam(model.parameters(), lr=LR, weight_decay=1e-5)
scheduler  = torch.optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode='min', factor=0.5, patience=5)
criterion  = nn.MSELoss()

train_losses, val_losses = [], []
best_val_loss = float('inf')
patience_count = 0
best_state = None

print('Starting training...')
for epoch in range(1, EPOCHS + 1):
    model.train()
    train_loss = 0.0
    for X_b, y30_b, y60_b in train_loader:
        X_b = X_b.to(DEVICE)
        y_b = torch.stack([y30_b, y60_b], dim=1).to(DEVICE)
        optimizer.zero_grad()
        pred = model(X_b)
        loss = criterion(pred, y_b)
        loss.backward()
        nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
        optimizer.step()
        train_loss += loss.item() * len(X_b)
    train_loss /= len(train_loader.dataset)

    model.eval()
    val_loss = 0.0
    with torch.no_grad():
        for X_b, y30_b, y60_b in val_loader:
            X_b = X_b.to(DEVICE)
            y_b = torch.stack([y30_b, y60_b], dim=1).to(DEVICE)
            val_loss += criterion(model(X_b), y_b).item() * len(X_b)
    val_loss /= len(val_loader.dataset)

    train_losses.append(train_loss)
    val_losses.append(val_loss)
    scheduler.step(val_loss)

    if val_loss < best_val_loss:
        best_val_loss = val_loss
        patience_count = 0
        best_state = {k: v.cpu().clone() for k, v in model.state_dict().items()}
        marker = ' <- best'
    else:
        patience_count += 1
        marker = ''

    if epoch % 5 == 0 or epoch == 1:
        print(f'Epoch {epoch:3d}/{EPOCHS} | Train: {train_loss:.6f} | Val: {val_loss:.6f}{marker}')

    if patience_count >= PATIENCE:
        print(f'Early stopping at epoch {epoch}.')
        break

model.load_state_dict(best_state)
print(f'Training complete. Best val loss: {best_val_loss:.6f}')

In [ ]:
# Cell 8: Evaluation Metrics
model.eval()
p30_list, p60_list, t30_list, t60_list = [], [], [], []
with torch.no_grad():
    for X_b, y30_b, y60_b in test_loader:
        pred = model(X_b.to(DEVICE)).cpu().numpy()
        p30_list.append(pred[:, 0])
        p60_list.append(pred[:, 1])
        t30_list.append(y30_b.numpy())
        t60_list.append(y60_b.numpy())

p30 = np.concatenate(p30_list)
p60 = np.concatenate(p60_list)
t30 = np.concatenate(t30_list)
t60 = np.concatenate(t60_list)

WL_MIN   = float(feat_min[0])
WL_RANGE = float(feat_range[0])
p30_m = p30 * WL_RANGE + WL_MIN
p60_m = p60 * WL_RANGE + WL_MIN
t30_m = t30 * WL_RANGE + WL_MIN
t60_m = t60 * WL_RANGE + WL_MIN

def compute_metrics(true_m, pred_m, horizon):
    mse  = mean_squared_error(true_m, pred_m)
    rmse = np.sqrt(mse)
    mae  = mean_absolute_error(true_m, pred_m)
    r2   = r2_score(true_m, pred_m)
    print(f'  {horizon}: RMSE={rmse*100:.2f}cm  MAE={mae*100:.2f}cm  NSE/R2={r2:.4f}')
    return {'MSE': mse, 'RMSE': rmse, 'MAE': mae, 'NSE_R2': r2}

print('Test Set Evaluation (in meters):')
m30 = compute_metrics(t30_m, p30_m, 'Horizon +30min')
m60 = compute_metrics(t60_m, p60_m, 'Horizon +60min')

test_metrics = {
    'horizon_30m': {k: float(v) for k, v in m30.items()},
    'horizon_60m': {k: float(v) for k, v in m60.items()},
}
with open('test_metrics.json', 'w') as f:
    json.dump(test_metrics, f, indent=2)
print('test_metrics.json saved.')

In [ ]:
# Cell 9: Training Curves Plot
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
fig.suptitle('SmartFlood 2026 LSTM Training Curves (2-Feature: no rainfall)', fontsize=13, fontweight='bold')

axes[0].plot(train_losses, label='Train Loss', color='#2196F3', linewidth=2)
axes[0].plot(val_losses,   label='Val Loss',   color='#FF5722', linewidth=2, linestyle='--')
axes[0].set_xlabel('Epoch'); axes[0].set_ylabel('MSE Loss')
axes[0].set_title('Training & Validation Loss'); axes[0].legend(); axes[0].grid(True, alpha=0.3); axes[0].set_yscale('log')

n_show = min(200, len(t30_m))
axes[1].plot(t30_m[:n_show], label='Actual',      color='#4CAF50', linewidth=2)
axes[1].plot(p30_m[:n_show], label='+30min Pred', color='#FF9800', linewidth=1.5, linestyle='--')
axes[1].plot(p60_m[:n_show], label='+60min Pred', color='#E91E63', linewidth=1.5, linestyle=':')
axes[1].axhline(1.0, color='gold',   linestyle=':', alpha=0.6)
axes[1].axhline(1.4, color='orange', linestyle=':', alpha=0.6)
axes[1].axhline(1.6, color='red',    linestyle=':', alpha=0.6)
axes[1].set_xlabel('Sample'); axes[1].set_ylabel('Water Level (m)')
axes[1].set_title('Prediction vs Actual (Test Set)'); axes[1].legend(); axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig('training_curves.png', dpi=150, bbox_inches='tight')
plt.show()
print('training_curves.png saved.')

In [ ]:
# Cell 10: Hydrograph Samples Plot
fig, axes = plt.subplots(2, 2, figsize=(14, 8))
fig.suptitle('SmartFlood 2026 Sample Hydrograph Predictions (2-Feature Model)', fontsize=13, fontweight='bold')
scenarios_to_plot = [('Typhoon Surge', 'typhoon'), ('Flash Flood', 'flash'), ('Moderate Rain', 'moderate'), ('Calm Baseline', 'baseline')]

for idx, (title, key) in enumerate(scenarios_to_plot):
    ax = axes[idx // 2][idx % 2]
    lvls = generate_hydrograph_scenario(96, key)
    surge = compute_surge_velocity(lvls)
    feat_n = ((np.column_stack([lvls, surge]).astype(np.float32)) - feat_min) / feat_range
    preds30, preds60, times = [], [], []
    model.cpu()
    with torch.no_grad():
        for i in range(SEQ_LEN, len(feat_n) - HORIZON_60M_STEPS):
            window = torch.tensor(feat_n[i - SEQ_LEN:i]).unsqueeze(0)
            out = model(window).numpy()[0]
            preds30.append(out[0] * WL_RANGE + WL_MIN)
            preds60.append(out[1] * WL_RANGE + WL_MIN)
            times.append(i)
    model.to(DEVICE)
    t_full = np.arange(len(lvls)) * 10
    t_pred = np.array(times) * 10
    ax.fill_between(t_full, 1.0, 1.4, alpha=0.07, color='yellow')
    ax.fill_between(t_full, 1.4, 1.6, alpha=0.07, color='orange')
    ax.fill_between(t_full, 1.6, 3.5, alpha=0.07, color='red')
    ax.plot(t_full, lvls,            color='#2196F3', label='Actual',    linewidth=2)
    ax.plot(t_pred, preds30, color='#FF9800', label='+30m Pred', linewidth=1.5, linestyle='--')
    ax.plot(t_pred, preds60, color='#E91E63', label='+60m Pred', linewidth=1.5, linestyle=':')
    ax.axhline(1.0, color='gold', linestyle=':', alpha=0.8, linewidth=1)
    ax.axhline(1.4, color='orange', linestyle=':', alpha=0.8, linewidth=1)
    ax.axhline(1.6, color='red', linestyle=':', alpha=0.8, linewidth=1)
    ax.set_title(title); ax.set_xlabel('Time (min)'); ax.set_ylabel('Water Level (m)')
    ax.set_ylim(0.28, min(3.5, lvls.max() + 0.2)); ax.legend(fontsize=8); ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig('hydrograph_samples.png', dpi=150, bbox_inches='tight')
plt.show()
print('hydrograph_samples.png saved.')

In [ ]:
# Cell 11: Export to ONNX
model.eval().cpu()
dummy_input = torch.zeros(1, SEQ_LEN, N_FEATURES)

torch.onnx.export(
    model, dummy_input, 'flood_lstm.onnx',
    input_names=['input'], output_names=['output'],
    dynamic_axes={'input': {0: 'batch_size'}, 'output': {0: 'batch_size'}},
    opset_version=17, verbose=False,
)

onnx.checker.check_model(onnx.load('flood_lstm.onnx'))
sess = ort.InferenceSession('flood_lstm.onnx')
result = sess.run(None, {'input': np.zeros((1, SEQ_LEN, N_FEATURES), dtype=np.float32)})[0]
print(f'ONNX export verified.')
print(f'  Input shape  : [1, {SEQ_LEN}, {N_FEATURES}]  (batch, seq_len, features)')
print(f'  Output shape : {result.shape}  = [batch, 2] = [+30min, +60min]')
print(f'  File size    : {os.path.getsize("flood_lstm.onnx") / 1024:.1f} KB')

In [ ]:
# Cell 12: Save scaler_params.json
scaler_params = {
    'method': 'minmax',
    'feature_order': ['water_level_m', 'surge_velocity'],
    'min':   feat_min.tolist(),
    'max':   feat_max.tolist(),
    'range': feat_range.tolist(),
    'target_source_feature': 'water_level_m',
    'target_source_index': 0,
    'sequence_length': SEQ_LEN,
    'horizons_minutes': [30, 60],
    'horizon_steps': [HORIZON_30M_STEPS, HORIZON_60M_STEPS],
    'timestep_minutes': TIMESTEP_MINUTES,
    'n_features': N_FEATURES,
    'notes': 'Apply x_norm = (x - min) / range per feature (order: water_level_m, surge_velocity). rainfall_rate REMOVED per adviser instruction. Invert output: y = y_norm * range[0] + min[0].'
}
with open('scaler_params.json', 'w') as f:
    json.dump(scaler_params, f, indent=2)
print('scaler_params.json saved.')
print(json.dumps(scaler_params, indent=2))

In [ ]:
# Cell 13: Download All Files
from google.colab import files
print('Downloading output files...')
print('Place these in: server/models/')
files.download('flood_lstm.onnx')
files.download('scaler_params.json')
files.download('test_metrics.json')
files.download('training_curves.png')
files.download('hydrograph_samples.png')

## After Downloading

Copy the 5 downloaded files to:
```
server/models/flood_lstm.onnx
server/models/scaler_params.json
server/models/test_metrics.json
server/models/training_curves.png
server/models/hydrograph_samples.png
```

| Parameter | Old (3-Feature) | New (2-Feature) |
|---|---|---|
| Features | [water_level_m, rain_rate_mmh, surge_velocity] | [water_level_m, surge_velocity] |
| Input tensor | [1, 6, 3] | [1, 6, 2] |
| Outputs | [+30min, +60min] | [+30min, +60min] |